## 10-714：作业 0

本次作业的目标，是让你快速了解一些在学习本课程之前就应该熟悉的概念和想法。本作业会要求你构建一个基础的 softmax 回归算法，以及一个简单的双层神经网络。你需要分别用原生 Python（使用 numpy 库）实现这些内容，并且针对 softmax 回归，还要用原生 C/C++ 实现。本作业也会带你走一遍把作业提交到自动评分系统的流程。在这个过程中，我们会给出一些关于如何实现不同函数的建议，但总体细节由你自己决定。不过我们要说明的是，在 Python 版本中，你应该大量使用 numpy 的线性代数调用：如果尝试显式写循环，通常会让代码比应有的速度慢很多。

**我们知道这份作业里有大量说明文字，尤其是开头部分，而真正需要写的代码相对较少。即便如此，_请_ 仔细通读这份说明的全部文本。这样做会帮助你理解我们组织作业的流程和理念，并且会极大影响你完成后续作业的能力。**

10-714 的所有作业代码开发都可以在 Google Colab 环境中完成。不过，我们不会在 Colab notebook 里大量使用实际代码块；你开发的大部分代码会写在（自动）下载到 Google Drive 的 `.py` 文件中，而 notebook 主要用于运行测试代码和提交代码到自动评分器的 shell 脚本（你也可以选择用它测试开发过程中的代码片段，但这不是必需的）。这是一种有点非标准的 Colab Notebook 用法（通常大家更像使用交互式编程环境那样，直接在 notebook 的代码单元中写代码）。不过，我们这样使用它的理由其实很直接：除了是一个不错的云端 notebook 环境之外，Colab 还提供了很方便的“标准”云端 GPU 系统，你可以快速启动它们。这会让你在后续开发某些代码，尤其是基于 CUDA 的代码时，不需要自己拥有实体 GPU，也不需要自己配置 CUDA 库。话虽如此，**你可以在任何你喜欢的环境中开发和提交代码**，只是我们不能保证对 Colab 之外的环境提供支持。

开始之前，请先在“File”菜单中选择“Save a copy in Drive”，**复制一份这个 notebook**，然后运行下面的代码块。这会把你的 Google Drive 文件夹加载到 Colab notebook 环境中，创建 `/10714/hw0` 目录，并把 HW0 的公开仓库 clone 到这个目录里。

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsyscourse/hw0.git
%cd /content/drive/MyDrive/10714/hw0

下一个单元会安装所需的库。

In [ ]:
!pip3 install --upgrade --no-deps git+https://github.com/dlsyscourse/mugrade.git
!pip3 install pybind11
!pip3 install numdifftools

## 问题 1：一个基础的 `add` 函数，以及测试/自动评分基础

为了说明这些作业和自动评分系统的工作流程，我们会使用一个简单的例子：实现一个 `add` 函数。注意，上面运行的命令会在你的 `10714/hw0` 目录中创建如下结构

    data/
        train-images-idx3-ubyte.gz
        train-labels-idx1-ubyte.gz
        t10k-images-idx3-ubyte.gz
        t10k-labels-idx1-ubyte.gz
    src/
        simple_ml.py
        simple_ml_ext.cpp
    tests/
        test_simple_ml.py
    Makefile
    
`data/` 目录包含本作业所需的数据（MNIST 数据集的一份副本）；`src/` 目录包含源文件，你会在其中写自己的实现；`tests/` 目录包含用于在本地评估你的解法并提交到自动评分器的测试；`Makefile` 文件是一个 makefile，用于编译代码（这和作业中的 C++ 部分有关）。

第一个作业问题要求你实现 `simple_ml.add()` 函数（这个非常简单的函数不会在别处使用，它只是一个例子，用来让你熟悉作业结构）。查看 `src/simple_ml.py` 文件时，你会看到 `add()` 函数的如下函数桩。

```python
def add(x, y):
    """ A trivial 'add' function you should implement to get used to the 
    autograder and submission system.  The solution to this problem is in
    the homework notebook.

    Args:
        x (Python number or numpy array)
        y (Python number or numpy array)

    Return:
        Sum of x + y
    """
    ### YOUR CODE HERE
    pass
    ### END YOUR CODE
```
每个文件中的 docstring 都定义了你的函数应该产生的预期输入/输出映射（你需要习惯仔细阅读，因为我们通常发现，提交中最常见的错误来源就是没有阅读规范）。希望你很容易看出这个函数应该如何实现。你只需要用正确的代码替换 `pass` 语句，也就是如下内容：

```python
def add(x, y):
    """ A trivial 'add' function you should implement to get used to the 
    autograder and submission system.  The solution to this problem is in the
    the homework notebook.

    Args:
        x (Python number or numpy array)
        y (Python number or numpy array)

    Return:
        Sum of x + y
    """
    ### YOUR CODE HERE
    return x + y
    ### END YOUR CODE
```
请在你的 `src/simple_ml.py` 文件中完成这个修改。

### 运行本地测试

现在你需要测试自己的代码是否能正常工作；如果可以，就把它提交到自动评分系统。本课程中，我们使用标准工具来运行代码单元测试，也就是 `pytest` 系统。当你在 `src/simple_ml.py` 文件中写好正确代码后，运行下面的命令。

In [9]:
!python3 -m pytest -k "add"

如果一切顺利，你会看到一个测试成功通过。要了解这个测试是如何工作的，请查看 `tests/test_simple_ml.py` 文件，特别是 `test_add()` 函数：

```python
def test_add():
    assert add(5,6) == 11
    assert add(3.2,1.0) == 4.2
    assert type(add(4., 4)) == float
    np.testing.assert_allclose(add(np.array([1,2]), np.array([3,4])),
                               np.array([4,6]))
```

这段代码会针对你实现的函数运行一组单元测试。如果函数实现正确，那么上面的所有断言都_应该_通过（也就是说，代码会无错误地执行）。如果你实现错了（比如把上面的 `x + y` 改成了 `x - y`），那么这些断言会失败，`pytest` 会指出对应的测试失败。

In [24]:
# in this example cell, we replaced "x + y" with "x - y" in simple_ml.add()
!python3 -m pytest -k "add"

============================= test session starts ==============================
platform darwin -- Python 3.7.3, pytest-4.3.1, py-1.8.0, pluggy-0.9.0
rootdir: /Users/zkolter/Dropbox/class/10-714/homework/hw0, inifile:
plugins: remotedata-0.3.1, openfiles-0.3.2, doctestplus-0.3.0, arraydiff-0.3
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py F                                                [100%]

=================================== FAILURES ===================================
___________________________________ test_add ___________________________________

    def test_add():
>       assert add(5,6) == 11
E       assert -1 == 11
E        +  where -1 = add(5, 6)

tests/test_simple_ml.py:16: AssertionError
==================== 1 failed, 5 deselected in 0.35 seconds ====================


如你所见，你会得到一个错误信息，指出断言失败的具体行号。然后你可以利用这个信息回到实现中调试。**你应该熟悉如何阅读和追踪测试文件，因为这是更好理解你的实现应该如何工作的方式。**

学会正确开发和使用单元测试，对现代软件开发至关重要。希望本课程的一个附带收获，是让你熟悉单元测试在软件开发中的典型用法。当然，这里并不完全如此，因为你不一定需要_自己编写_测试也能通过这些问题；但你_应该_熟悉如何阅读我们提供的测试文件，从而理解函数应有的行为。不过，我们也_非常_鼓励你为自己的实现编写额外测试，尤其是当你的代码通过了本地测试，但提交后似乎仍然失败时。

最后再快速说明一点。如果你习惯用 print 语句调试代码，请注意 **pytest 默认会捕获所有输出**。你可以向 pytest 传入 `-s` 标志来禁用这个行为，让测试在所有情况下都显示全部输出。

### 提交到自动评分器

现在你已经通过了单元测试，是时候提交你的解法进行自动评分了。本课程的自动评分使用由课程教师编写的自定义应用。要开始自动评分流程，请访问 https://mugrade.org/courses/1?enroll=664EqzbhsV5p99rbBYf0，并通过 Google 登录，登录时**使用你的 Andrew 邮箱**。

创建账号后，点击 Deep Learning Systems 进入课程。在页面右上角附近点击 **Show API Key**，并复制对应的 key。这个 key 与_你的提交_绑定，任何拿到这个 key 的人都可以提交你的作业；因此，你不应该把这个 key 分享给任何人，只应该在下面用于提交自己的代码。拿到这个 key 后，运行下面的命令。

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "add"

运行这个命令会把你的 `add` 函数提交到 mugrade 自动评分系统。要了解其内部工作方式，请再次查看 `tests/test_simply_ml.py` 文件，这次看 `test_add` 函数正下方的 `submit_add()` 函数。

```python
def submit_add():
    mugrade.submit(add(1,2))
    mugrade.submit(add(4.5, 3.2))
    mugrade.submit(type(add(3,2)))
    mugrade.submit(add(np.array([1.,2.]), np.array[5,6]))
```

这段代码看起来有点像上面的单元测试，但它不是使用断言，而是调用 `mugrade.submit()`。这些调用会分别在不同输入上计算 `add` 函数，然后把结果发送到 mugrade 服务器。服务器会把你的函数输出与正确输出进行比较（正确输出只存储在服务器上，而不是本地，所以你无法提前知道正确答案），并相应更新你的作业分数。如果你已经登录 mugrade 系统，可以进入 "Homework 0" 作业页面查看更新后的成绩（如果你已经停留在该页面，必要时刷新页面）。

**重要说明：** 如果你熟悉自动评分系统，可能会注意到 mugrade 的工作方式和大多数系统略有不同。在大多数自动评分系统中，你会编写代码以通过本地测试（如果幸运的话，有些课程甚至不提供本地测试），然后把代码打包提交到自动评分系统。自动评分服务器会解包并执行你的代码，在一些（对你来说未知的）测试用例上运行它。Mugrade 不同：`submit_add` 函数是在_你的系统_上运行的（例如你正在使用的 Colab 环境），只有函数调用的_结果_会被发送到服务器。

这种设置背后的理由很微妙，但很重要。本课程要求你开发一个相当复杂的系统，它会让你的代码运行在 GPU 等专用硬件上，也可能运行训练真实神经网络架构的较长测试。如果无法在自动评分测试用例上调试代码执行过程，实践中会带来很大的挑战，更不用说服务器容量问题，以及通常在评分截止时间前出现的变慢问题。把计算移到本地，意味着你可以更清楚地了解自己的代码在自动评分测试用例上实际如何运行，这对调试非常有价值。对于类似课程，论坛中_最常见_的帖子大概就是“我的代码通过了所有本地测试，但在自动评分器上失败”；虽然这种情况在 mugrade 中_仍然_可能发生，但至少你可以单步跟踪自己的代码执行，查看它在_哪里_以及_如何_失败。并且，由于服务器只需要对你提交的输入和正确输出做简单检查，服务器会很简单，即使在提交截止时间附近，你也会立即得到自动评分器的反馈。

这类评分系统的_缺点_是存在作弊的可能。因为你可以完全控制自动评分测试用例在本地执行代码的过程，所以理论上你可以直接找出正确答案，并在没有真正实现所需代码的情况下返回它们。比如在本作业后面，你需要在初始 Python 实现之外，再写一个 C++ 版本的 softmax 回归。完全可以把自动评分器改成使用你的 Python 实现，而不是 C++ 实现，并且仍然通过测试。即便如此，**请不要以任何方式试图绕过自动评分系统**。我们设计这个系统的目标，是让你真正更容易调试和开发代码，让体验更接近“真实”的开发流程，而不是变成“弄清楚为什么自动评分器无法编译你的某种 CUDA 代码变体”。为了解决这一点，除了提交自动评分测试外，当你完成提交时，还需要把你的解法中 `/src` 目录的 `.tar.gz` 文件上传到 mugrade 系统。我们相信这不会成为问题，但如果确实有任何疑问，助教始终可以验证你的代码是否产生这些结果（他们也会使用 MOSS 等标准抄袭检测系统进行检查）。

## 问题 2：加载 MNIST 数据

现在你已经熟悉了自动评分系统，可以在 `src/simple_ml.py` 文件中下一个需要实现的函数 `parse_mnist_data()` 上试一试。下面是该文件中的函数声明（我们通常不会再次完整走一遍这个过程，但这里会再演示一次）。

```python
def parse_mnist(image_filename, label_filename):
    """ Read an images and labels file in MNIST format.  See this page:
    http://yann.lecun.com/exdb/mnist/ for a description of the file format.

    Args:
        image_filename (str): name of gzipped images file in MNIST format
        label_filename (str): name of gzipped labels file in MNIST format

    Returns:
        Tuple (X,y):
            X (numpy.ndarray[np.float32]): 2D numpy array containing the loaded 
                data.  The dimensionality of the data should be 
                (num_examples x input_dim) where 'input_dim' is the full 
                dimension of the data, e.g., since MNIST images are 28x28, it 
                will be 784.  Values should be of type np.float32, and the data 
                should be normalized to have a minimum value of 0.0 and a 
                maximum value of 1.0 (i.e., scale original values of 0 to 0.0 
                and 255 to 1.0).

            y (numpy.ndarray[dtype=np.uint8]): 1D numpy array containing the
                labels of the examples.  Values should be of type np.uint8 and
                for MNIST will contain the values 0-9.
    """
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE
```

希望你现在已经熟悉了 docstring 的作用，并且对如何实现这个函数有了一些想法。首先，请访问 http://yann.lecun.com/exdb/mnist/ 或这个备用[链接](https://web.archive.org/web/20220509025752/http://yann.lecun.com/exdb/mnist/)（页面底部），阅读 MNIST 数据的二进制格式。然后编写一个加载器，读取这种类型的文件，并按照 docstring 中的规范返回 numpy 数组（如果实现时遇到问题，一定要仔细阅读 docstring）。我们建议你使用 Python 的 `struct` 模块（以及 `gzip` 模块，当然还有 `numpy` 本身）来实现这个函数。

当你实现好该函数后，运行本地单元测试。

In [5]:
!python3 -m pytest -k "parse_mnist"

然后把你的代码提交到 mugrade。

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "parse_mnist"

## 问题 3：Softmax 损失

按照 `src/simple_ml.py` 中 `softmax_loss()` 函数的定义，实现 softmax（又称 cross-entropy，交叉熵）损失。回顾一下（希望这只是复习，但我们也会在 9/1 的课堂上讲到），对于一个可以取值为 $y \in \{1,\ldots,k\}$ 的多类别输出，softmax 损失接收一个 logits 向量 $z \in \mathbb{R}^k$ 和真实类别 $y \in \{1,\ldots,k\}$，并返回如下定义的损失：
$$
\ell_{\mathrm{softmax}}(z, y) = \log\sum_{i=1}^k \exp z_i - z_y.
$$

注意，正如 docstring 所描述的，`softmax_loss()` 接收一个 logits 的_二维数组_（也就是一批不同样本各自的 $k$ 维 logits），再加上对应的一维真实标签数组，并且应该输出整批数据上的_平均_ softmax 损失。注意，要正确完成这一点，你_不应该_使用任何循环，而应该完全使用 numpy 的向量化操作进行计算（为了让你有个预期，我们可以说明一下，我们的参考解法只有一行代码）。

注意，在“真实”的 softmax 损失实现中，你会希望缩放 logits 来防止数值溢出，但这里我们不需要担心这一点（即使你不处理这个问题，本作业的其余部分也能正常工作）。下面的代码会运行测试用例。

In [10]:
!python3 -m pytest -k "softmax_loss"

然后运行提交。

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "softmax_loss"

## 问题 4：Softmax 回归的随机梯度下降

在这个问题中，你将为（线性）softmax 回归实现随机梯度下降（SGD）。换句话说，如 9/1 课堂中所讨论的，我们会考虑一个假设函数，它通过下面的函数把 $n$ 维输入映射到 $k$ 维 logits：
$$
h(x) = \Theta^T x
$$
其中 $x \in \mathbb{R}^n$ 是输入，$\Theta \in \mathbb{R}^{n \times k}$ 是模型参数。给定数据集 $\{(x^{(i)} \in \mathbb{R}^n, y^{(i)} \in \{1,\ldots,k\})\}$，其中 $i=1,\ldots,m$，与 softmax 回归相关的优化问题可以写成：
$$
\mathop{\mathrm{minimize}}_{\Theta} \; \frac{1}{m} \sum_{i=1}^m \ell_{\mathrm{softmax}}(\Theta^T x^{(i)}, y^{(i)}).
$$

回顾课堂内容，线性 softmax 目标的梯度为：
$$
\nabla_\Theta \ell_{\mathrm{softmax}}(\Theta^T x, y) = x (z - e_y)^T
$$
其中
$$
z = \frac{\exp(\Theta^T x)}{1^T \exp(\Theta^T x)} \equiv \operatorname{normalize}(\exp(\Theta^T x))
$$
（也就是说，$z$ 就是归一化后的 softmax 概率），而 $e_y$ 表示第 $y$ 个单位基向量，也就是一个除了第 $y$ 个位置为 1、其余位置全为 0 的向量。

我们也可以使用课堂中讨论过的更紧凑记号来写它。具体来说，令 $X \in \mathbb{R}^{m \times n}$ 表示某 $m$ 个输入的设计矩阵（可以是整个数据集，也可以是一个 minibatch），令 $y \in \{1,\ldots,k\}^m$ 表示对应的标签向量，并重载 $\ell_{\mathrm{softmax}}$ 这个记号，使其表示平均 softmax 损失，那么有：
$$
\nabla_\Theta \ell_{\mathrm{softmax}}(X \Theta, y) = \frac{1}{m} X^T (Z - I_y)
$$
其中
$$
Z = \operatorname{normalize}(\exp(X \Theta)) \quad \text{(normalization applied row-wise)}
$$
表示 logits 矩阵，而 $I_y \in \mathbb{R}^{m \times k}$ 表示把 $y$ 中各个标签对应的 one-hot 基向量拼接起来得到的矩阵。

使用这些梯度，实现 `softmax_regression_epoch()` 函数。该函数使用指定的学习率/步长 `lr` 和 minibatch 大小 `batch`，运行一个 SGD epoch（对数据集的一次遍历）。如 docstring 所述，你的函数应该原地修改 `Theta` 数组。实现后，运行测试。

In [ ]:
!python3 -m pytest -k "softmax_regression_epoch and not cpp"

然后运行提交。

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "softmax_regression_epoch and not cpp"

### 使用 softmax 回归训练 MNIST

虽然这不是测试的一部分，但既然你已经写好了这段代码，也可以尝试用 SGD 训练一个完整的 MNIST 线性分类器。为此，你可以使用 `src/simple_ml.py` 文件中的 `train_softmax()` 函数（我们已经为你写好了这个函数，所以你不需要自己写，不过你可以看看它在做什么）。

你可以用下面的代码看看它如何工作。作为参考，如下所示，我们的实现在 Colab 上运行约 3 秒，并达到 7.97% 的错误率。

In [26]:
import sys
sys.path.append("src/")
from simple_ml import train_softmax, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")

train_softmax(X_tr, y_tr, X_te, y_te, epochs=10, lr=0.2, batch=100)

| Epoch | Train Loss | Train Err | Test Loss | Test Err |
|     0 |    0.35134 |   0.10182 |   0.33588 |  0.09400 |
|     1 |    0.32142 |   0.09268 |   0.31086 |  0.08730 |
|     2 |    0.30802 |   0.08795 |   0.30097 |  0.08550 |
|     3 |    0.29987 |   0.08532 |   0.29558 |  0.08370 |
|     4 |    0.29415 |   0.08323 |   0.29215 |  0.08230 |
|     5 |    0.28981 |   0.08182 |   0.28973 |  0.08090 |
|     6 |    0.28633 |   0.08085 |   0.28793 |  0.08080 |
|     7 |    0.28345 |   0.07997 |   0.28651 |  0.08040 |
|     8 |    0.28100 |   0.07923 |   0.28537 |  0.08010 |
|     9 |    0.27887 |   0.07847 |   0.28442 |  0.07970 |


## 问题 5：双层神经网络的 SGD

现在你已经为线性分类器写好了 SGD，我们来考虑一个简单双层神经网络的情形。具体来说，对于输入 $x \in \mathbb{R}^n$，我们考虑如下形式的双层神经网络（不含偏置项）：
$$
z = W_2^T \mathrm{ReLU}(W_1^T x)
$$
其中 $W_1 \in \mathbb{R}^{n \times d}$ 和 $W_2 \in \mathbb{R}^{d \times k}$ 表示网络权重（该网络具有一个 $d$ 维隐藏单元），而 $z \in \mathbb{R}^k$ 表示网络输出的 logits。我们仍然使用 softmax / cross-entropy 损失，也就是说，我们希望求解如下优化问题：
$$
\mathop{\mathrm{minimize}}_{W_1, W_2} \;\; \frac{1}{m} \sum_{i=1}^m \ell_{\mathrm{softmax}}(W_2^T \mathrm{ReLU}(W_1^T x^{(i)}), y^{(i)}).
$$
或者，重载记号以描述矩阵 $X \in \mathbb{R}^{m \times n}$ 的 batch 形式，也可以写成：
$$
\mathop{\mathrm{minimize}}_{W_1, W_2} \;\; \ell_{\mathrm{softmax}}(\mathrm{ReLU}(X W_1) W_2, y).
$$

利用链式法则，我们可以推导出这个网络的反向传播更新（我们会在 9/8 的课堂上简要介绍这些内容，但这里也给出最终形式，方便你实现）。具体来说，令
$$
\begin{aligned}
Z_1 \in \mathbb{R}^{m \times d} & = \mathrm{ReLU}(X W_1) \\
G_2 \in \mathbb{R}^{m \times k} & = \operatorname{normalize}(\exp(Z_1 W_2)) - I_y \\
G_1 \in \mathbb{R}^{m \times d} & = \mathrm{1}\{Z_1 > 0\} \circ (G_2 W_2^T)
\end{aligned}
$$
其中 $\mathrm{1}\{Z_1 > 0\}$ 是一个二值矩阵，它的每个元素根据 $Z_1$ 中对应项是否严格为正而取 0 或 1；$\circ$ 表示逐元素乘法。那么目标函数的梯度为
$$
\begin{aligned}
\nabla_{W_1} \ell_{\mathrm{softmax}}(\mathrm{ReLU}(X W_1) W_2, y) & = \frac{1}{m} X^T G_1  \\
\nabla_{W_2} \ell_{\mathrm{softmax}}(\mathrm{ReLU}(X W_1) W_2, y) & = \frac{1}{m} Z_1^T G_2.
\end{aligned}
$$

**注意：** 如果这些精确方程的细节对你来说有点晦涩（特别是在 9/8 课程之前），不必太担心。它们_只是_双层 ReLU 网络的标准反向传播方程：$Z_1$ 项计算“前向”传播，而 $G_2$ 和 $G_1$ 项表示反向传播。但根据你使用的神经网络记号、你组织损失函数的具体方式、你之前是否用矩阵形式推导过这些内容等，更新的精确形式可能会有所不同。如果这些记号让你觉得和过去见过的深度网络有些相似，并且在 9/8 课程之后更有意义，那么作为背景知识已经足够了（毕竟，某种程度上，深度学习系统的整个_重点_就是我们不需要一直处理这些手工计算）。但如果这些概念对你来说_完全_陌生，那么在学习本课程之前，可能更适合先单独上一门机器学习和神经网络课程，或者至少意识到本课程中会有大量需要补上的背景内容。

使用这些梯度，现在在 `src/simple_ml.py` 文件中编写 `nn_epoch()` 函数。和前一个问题一样，你的解法应该原地修改 `W1` 和 `W2` 数组。实现该函数后，运行下面的测试。请务必按照上面表达式所示使用矩阵操作来实现该函数：这会比尝试使用循环_快得多_，也更高效，而且需要的代码少得多。

In [ ]:
!python3 -m pytest -k "nn_epoch"

最后提交到自动评分器。

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "nn_epoch"

### 训练完整神经网络

和前面一样，虽然这不是通过自动评分器的严格必要条件，但看看你能用自己的神经网络函数把 MNIST 分类器训练到什么程度，还是很有意思的。类似 softmax 回归的情形，`simple_ml.py` 文件中有一个 `train_nn()` 函数，你可以用它通过多轮 SGD 训练这个双层网络。下面这段代码会训练一个具有 400 个隐藏单元的双层网络。

In [25]:
import sys

# Reload the simple_ml module which has been cached from the earlier experiment
import importlib
import simple_ml
importlib.reload(simple_ml)

sys.path.append("src/")
from simple_ml import train_nn, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")
train_nn(X_tr, y_tr, X_te, y_te, hidden_dim=400, epochs=20, lr=0.2)

| Epoch | Train Loss | Train Err | Test Loss | Test Err |
|     0 |    0.15324 |   0.04697 |   0.16305 |  0.04920 |
|     1 |    0.09854 |   0.02923 |   0.11604 |  0.03660 |
|     2 |    0.07392 |   0.02163 |   0.09750 |  0.03200 |
|     3 |    0.06006 |   0.01757 |   0.08825 |  0.02960 |
|     4 |    0.04869 |   0.01368 |   0.08147 |  0.02620 |
|     5 |    0.04061 |   0.01093 |   0.07698 |  0.02380 |
|     6 |    0.03494 |   0.00915 |   0.07446 |  0.02320 |
|     7 |    0.03027 |   0.00758 |   0.07274 |  0.02320 |
|     8 |    0.02674 |   0.00650 |   0.07103 |  0.02240 |
|     9 |    0.02373 |   0.00552 |   0.06989 |  0.02150 |
|    10 |    0.02092 |   0.00477 |   0.06870 |  0.02130 |
|    11 |    0.01914 |   0.00403 |   0.06837 |  0.02130 |
|    12 |    0.01705 |   0.00325 |   0.06748 |  0.02150 |
|    13 |    0.01541 |   0.00272 |   0.06688 |  0.02130 |
|    14 |    0.01417 |   0.00232 |   0.06657 |  0.02090 |
|    15 |    0.01282 |   0.00195 |   0.06591 |  0.02040 |
|    16 |    0

对于我们的实现来说，这在 Colab 上大约需要 30 秒运行时间；如上所示，它在 MNIST 上达到了 1.89\% 的错误率。对于不到 20 行左右的代码来说，还不错……

## 问题 6：C++ 中的 softmax 回归

本作业的最后一个问题要求你实现和问题 4 中相同的函数，也就是运行线性 softmax 回归单个 epoch 的函数。但这一次，你要使用 C++ 实现，而不是 Python 实现。（严格来说，这里的实际实现更像原始 C，但我们会使用 C++ 特性，通过 [pybind11](https://pybind11.readthedocs.io) 库构建到 Python 的接口；你在后续作业中也会使用它在 C++ 和 Python 之间建立接口。虽然还有其他替代方案，但 pybind11 作为接口库相当不错，因为它是 header-only 的，并且允许你在单个 C++ 源文件中实现完整的 Python/C++ 接口。）

你要实现内容的 C++ 文件是 `src/simple_ml_ext.cpp`。我们来看一下文件中的相关部分。你需要在下面这个函数中实现自己的代码：

```cpp
void softmax_regression_epoch_cpp(const float *X, const unsigned char *y, 
								  float *theta, size_t m, size_t n, size_t k, 
								  float lr, size_t batch)
{
    /**
     * A C++ version of the softmax regression epoch code.  This should run a 
     * single epoch over the data defined by X and y (and sizes m,n,k), and
     * modify theta in place.  Your function will probably want to allocate
     * (and then delete) some helper arrays to store the logits and gradients.
     * 
     * Args:
     *     X (const float *): pointer to X data, of size m*n, stored in row 
     *          major (C) format
     *     y (const unsigned char *): pointer to y data, of size m
     *     theta (float *): pointer to theta data, of size n*k, stored in row
     *          major (C) format
     *     m (size_t): number of examples
     *     n (size_t): input dimension
     *     k (size_t): number of classes
     *     lr (float): learning rate / SGD step size
     *     batch (int): SGD minibatch size
     * 
     * Returns:
     *     (None)
     */

    /// YOUR CODE HERE
    
    /// END YOUR CODE
}
```
我们稍微拆解一下这个函数的参数。这个函数本质上对应 Python 实现，但因为我们操作的是数组数据的原始指针，而不是某种更高级的“矩阵”数据结构，所以需要传入一些额外参数。具体来说，`X`、`y` 和 `theta` 是指向上一部分中对应 numpy 数组原始数据的指针；对于二维数组，它们以 C 风格（row-major，行主序）格式存储，意思是 $X$ 的第一行会作为 `X` 中最前面的若干字节连续存储，然后是第二行，依此类推（这与 _column major_，列主序，相对；列主序会先连续存储矩阵的第一_列_，然后是第二列，依此类推）。我们还假设数据中没有 padding；也就是说，第二行紧跟在第一行之后，中间没有为了对齐到某个内存边界而额外加入的字节（所有这些问题都会在课程后续讨论中提到，但现在先避免）。当然，由于传入函数的只有原始数据，为了知道底层矩阵的实际大小，我们还需要把这些大小显式传入函数，这正是 `m`、`n` 和 `k` 参数提供的信息。

作为说明，注意由于 `X` 表示一个行主序的 $m \times n$ 矩阵，如果我们想访问 $X$ 的 $(i,j)$ 元素（第 $i$ 行第 $j$ 列的元素），应该使用如下索引：
```cpp
X[i*n + j]
```
也就是说，因为 `X` 有 $n$ 列，并且会按行连续存储各列，所以要访问第 $i$ 行，需要先访问 `X[i*n]` 这个元素；如果想访问这一行中的第 $j$ 个元素，就使用上面的表达式。同样的逻辑也适用于 `theta` 矩阵，但重要的是，因为 `theta` 是一个 $n \times k$ 矩阵，要访问它的 $(i,j)$ 元素，你应该使用如下索引：
```cpp
theta[i*k + j]
```
与 Python 不同，在 C++ 中像这样直接访问内存时，你需要非常小心，并且要非常熟悉这种记号（或者构建额外的数据结构来帮助你以更直观的方式访问内容，不过在本作业中你应该只坚持使用原始索引）。


实现中的第二个重要部分，是实际提供 Python 接口的 pybind11 代码：
```cpp
PYBIND11_MODULE(simple_ml_ext, m) {
    m.def("softmax_regression_epoch_cpp", 
    	[](py::array_t<float, py::array::c_style> X, 
           py::array_t<unsigned char, py::array::c_style> y, 
           py::array_t<float, py::array::c_style> theta,
           float lr,
           int batch) {
        softmax_regression_epoch_cpp(
        	static_cast<const float*>(X.request().ptr),
            static_cast<const unsigned char*>(y.request().ptr),
            static_cast<float*>(theta.request().ptr),
            X.request().shape[0],
            X.request().shape[1],
            theta.request().shape[1],
            lr,
            batch
           );
    },
    py::arg("X"), py::arg("y"), py::arg("theta"), 
    py::arg("lr"), py::arg("batch"));
}
```
这段代码已经在文件中提供给你了，你完全不应该修改它。不过对于好奇的同学，这段代码本质上只是从提供的输入中抽取原始指针（使用 pybind 的 numpy 接口），然后调用对应的 `softmax_regression_epoch_cpp` 函数。

基于这些背景，实现 `softmax_regression_epoch_cpp`，使其完成与你的 Python 实现相同的更新。注意，由于你只是在访问原始数据，所以需要手动执行所有矩阵-向量乘法，而不是依赖 numpy 帮你完成所有矩阵操作（**注意：本作业不要使用 Eigen 这样的外部矩阵库，而是自己写乘法……这是一个相对简单的乘法**）。完成后，你可以使用下面的命令测试实现。

In [28]:
!make
!python3 -m pytest -k "softmax_regression_epoch_cpp"

c++ -O3 -Wall -shared -std=c++11 -fPIC -undefined dynamic_lookup $(python3 -m pybind11 --includes) src/simple_ml_ext.cpp -o src/simple_ml_ext.so
============================= test session starts ==============================
platform darwin -- Python 3.7.3, pytest-4.3.1, py-1.8.0, pluggy-0.9.0
rootdir: /Users/zkolter/Dropbox/class/10-714/homework/hw0, inifile:
plugins: remotedata-0.3.1, openfiles-0.3.2, doctestplus-0.3.0, arraydiff-0.3
collected 6 items / 5 deselected / 1 selected                                  

tests/test_simple_ml.py .                                                [100%]

==================== 1 passed, 5 deselected in 0.49 seconds ====================


注意，和之前的代码不同，我们必须先实际编译 C++ 扩展，才能运行并测试它。只要代码中有 C++ 组件，就需要这样做；但在所有这类情况下，我们都会提供 Makefile，用于编译所有相关函数并包含必要的库/头文件。最后，我们也把结果提交到 mugrade。

In [ ]:
!python3 -m mugrade submit YOUR_GRADER_KEY_HERE hw0 -k "softmax_regression_epoch_cpp"

### 使用 C++ 版本训练完整的 softmax 回归分类器

最后，让我们尝试使用这个“直接内存访问”的 C++ 版本训练整个 softmax 回归分类器。如果前面的 Python 版本花了约 3 秒，那这个版本应该快得飞起，对吧？

In [30]:
import sys
sys.path.append("src/")

# Reload the simple_ml module to include the newly-compiled C++ extension
import importlib
import simple_ml
importlib.reload(simple_ml)

from simple_ml import train_softmax, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")

train_softmax(X_tr, y_tr, X_te, y_te, epochs=10, lr = 0.2, batch=100, cpp=True)

| Epoch | Train Loss | Train Err | Test Loss | Test Err |
|     0 |    0.35134 |   0.10182 |   0.33588 |  0.09400 |
|     1 |    0.32142 |   0.09268 |   0.31086 |  0.08730 |
|     2 |    0.30802 |   0.08795 |   0.30097 |  0.08550 |
|     3 |    0.29987 |   0.08532 |   0.29558 |  0.08370 |
|     4 |    0.29415 |   0.08323 |   0.29215 |  0.08230 |
|     5 |    0.28981 |   0.08182 |   0.28973 |  0.08090 |
|     6 |    0.28633 |   0.08085 |   0.28793 |  0.08080 |
|     7 |    0.28345 |   0.07997 |   0.28651 |  0.08040 |
|     8 |    0.28100 |   0.07923 |   0.28537 |  0.08010 |
|     9 |    0.27887 |   0.07847 |   0.28442 |  0.07970 |


正如预期，数字结果和 Python 版本完全一致，而代码却……慢了大约 5 倍？！这是怎么回事？其实，你很可能为 C++ 版本写的“手动”矩阵乘法代码极其低效。虽然 Python 本身是一种较慢的解释型语言，但 numpy 背后的矩阵乘法是用 C（或者更可能是 Fortran，信不信由你）写的，并且经过了高度优化，可以利用向量操作、不同处理器的缓存层次结构，以及高效数值运算所必需的其他特性。我们会在后续课程中更深入地讲这些细节，你甚至会写一个矩阵库，使它至少在某些特殊情况下可以相对高效地执行这些操作……老实说，一般情况下想超过 numpy 并不容易。

不过现在，只要你的代码复现了 Python 的行为，你就完成了本作业，并且可以准备进入下一部分：自动微分。